# Treinamento SVM - Variações de Balanceamento (Via Pipeline)
Este notebook utiliza **Pipelines** para garantir que o balanceamento de dados (Over/Undersampling) ocorra corretamente dentro da validação cruzada, sem vazamento de dados e respeitando os grupos (Cachorros).

### Instruções:
1. Configure a estratégia em `BALANCE_STRATEGY`.
2. Rode todas as células.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

try:
    from imblearn.pipeline import Pipeline 
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.over_sampling import RandomOverSampler
    print("Biblioteca imbalanced-learn carregada com sucesso.")
except ImportError:
    raise ImportError("Você precisa instalar a biblioteca: pip install imbalanced-learn")

repo_root = Path("../../").resolve() 
file_path = repo_root / "models" / "data" / "DogFeatures.csv"

In [ ]:
# ==========================================
# === CONFIGURAÇÃO DO EXPERIMENTO ===
# ==========================================

# Opções:
# 'none'         -> Sem balanceamento (Padrão)
# 'class_weight' -> SVM penaliza erros nas classes menores (Recomendado)
# 'under'        -> Remove exemplos das classes majoritárias
# 'over'         -> Duplica exemplos das classes minoritárias

BALANCE_STRATEGY = 'class_weight' 

print(f"Estratégia Selecionada: {BALANCE_STRATEGY.upper()}")

In [ ]:
if not file_path.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {file_path}")

df = pd.read_csv(file_path)
groups = df["DogID"]

cols_to_drop = ["DogID", "label", "Breed", "Gender", "NeuteringStatus", "TestNum", "t_dt"]
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

X = df.drop(columns=cols_to_drop)
y = df["label"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)
target_names = le.classes_

print(f"Dados carregados. Shape: {X.shape}")

In [ ]:
steps = []

if BALANCE_STRATEGY == 'under':
    steps.append(('sampler', RandomUnderSampler(random_state=42)))
elif BALANCE_STRATEGY == 'over':
    steps.append(('sampler', RandomOverSampler(random_state=42)))

steps.append(('scaler', StandardScaler()))

svm_args = {}
if BALANCE_STRATEGY == 'class_weight':
    svm_args['class_weight'] = 'balanced'

steps.append(('svm', SVC(**svm_args)))

pipeline = Pipeline(steps)
print("Pipeline construído:", steps)

In [ ]:
logo = LeaveOneGroupOut()

param_grid = [
    {'svm__C': [1, 10], 'svm__kernel': ['linear']},
    {'svm__C': [1, 10], 'svm__kernel': ['rbf'], 'svm__gamma': ['scale']}
]

grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=logo, 
    scoring='accuracy', 
    verbose=1,
    n_jobs=-1
)

print(f"Iniciando Grid Search com estratégia: {BALANCE_STRATEGY}...")

In [ ]:
grid_search.fit(X, y_encoded, groups=groups)

print(f"Melhor Score: {grid_search.best_score_:.4f}")
print(f"Melhores Params: {grid_search.best_params_}")

In [ ]:
print("Gerando predições com validação cruzada...")
best_model = grid_search.best_estimator_

# O cross_val_predict com Pipeline respeita o data leak
y_pred = cross_val_predict(best_model, X, y_encoded, groups=groups, cv=logo, n_jobs=-1)

print(classification_report(y_encoded, y_pred, target_names=target_names))

In [ ]:
cm = confusion_matrix(y_encoded, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', 
            xticklabels=target_names, 
            yticklabels=target_names)
plt.title(f'Matriz de Confusão - {BALANCE_STRATEGY.upper()}')

img_name = f"svm_cm_{BALANCE_STRATEGY}.png"
output_dir = repo_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(output_dir / img_name)
print(f"Imagem salva: {img_name}")

results_df = pd.DataFrame(grid_search.cv_results_)
csv_name = f"svm_results_{BALANCE_STRATEGY}.csv"
results_df.to_csv(output_dir / csv_name, index=False)
print(f"CSV salvo: {csv_name}")